# Biomedical NER: NCBI Disease → BC5CDR Transfer Learning

**Pipeline (from architecture diagram):**

- **Dataset A (NCBI Disease):** preprocess → BIO tag → subword align → fine-tune transformer NER → checkpoint
- **Dataset B (BC5CDR):** same pipeline → inference with checkpoint → span reconstruction → entity-level evaluation
- **Research questions answered:**
  - RQ1 – Per-category F1 (entity subtypes)
  - RQ2 – Learning curves (loss + F1 per epoch)
  - RQ3 – Error patterns (FP / FN / boundary errors)

In [18]:
# Install PyTorch with CUDA 12.8 support (RTX 5080 / Blackwell requires cu128)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -q
%pip install transformers datasets seqeval accelerate ipywidgets sentencepiece -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import os
import re
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple, Set

import numpy as np
import matplotlib.pyplot as plt
import torch
from dotenv import load_dotenv 
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from datasets import Dataset
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

load_dotenv()

token = os.getenv("HF_TOKEN")
# ── Paths ──────────────────────────────────────────────────────────────────────
WORKSPACE   = Path(".")
NCBI_DIR    = WORKSPACE / "ncbi_desease_data"
BC5CDR_DIR  = WORKSPACE / "BC5CDR"

# ── Models ─────────────────────────────────────────────────────────────────────
MODELS = {
    "bert-base": "bert-base-cased",
    "biobert":   "dmis-lab/biobert-base-cased-v1.1",
}

# ── Training hyper-parameters ──────────────────────────────────────────────────
MAX_LENGTH  = 512
BATCH_TRAIN = 16
BATCH_EVAL  = 32
LR          = 2e-5
EPOCHS      = 5
PATIENCE    = 2

# ── Label scheme ───────────────────────────────────────────────────────────────
LABEL2ID = {"O": 0, "B-Disease": 1, "I-Disease": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("Imports and config loaded.")
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")


Imports and config loaded.
PyTorch 2.11.0+cu128  |  CUDA available: True


## 1. Data Loading and Parsing

Both datasets use **PubTator format**:
```
PMID|t|Title text
PMID|a|Abstract text
PMID  start  end  mention  type  MeSH_ID
```
Character offsets are over `title + '\n' + abstract`.

- **NCBI Disease** annotation types: `SpecificDisease`, `DiseaseClass`, `Modifier`, `CompositeMention`
- **BC5CDR** annotation types: `Chemical`, `Disease` (plus `CID` relation lines — skipped here)

In [20]:
def parse_pubtator(filepath: Path, keep_types=None) -> List[Dict]:
    """Parse a PubTator file into a list of document dicts.

    Each doc: {pmid, title, abstract, text, annotations}
    Each annotation: {start, end, text, type}

    Args:
        filepath:   path to a .txt PubTator file
        keep_types: set of entity type strings to retain (None = keep all).
                    CID / relation lines are always skipped.
    """
    documents = []
    with open(filepath, "r", encoding="utf-8") as fh:
        content = fh.read()

    for block in re.split(r"\n\n+", content.strip()):
        lines = block.strip().splitlines()
        if not lines:
            continue

        doc = {"pmid": None, "title": "", "abstract": "", "text": "", "annotations": []}

        for line in lines:
            if "|t|" in line:
                pmid, _, title = line.partition("|t|")
                doc["pmid"] = pmid.strip()
                doc["title"] = title.strip()
            elif "|a|" in line:
                _, _, abstract = line.partition("|a|")
                doc["abstract"] = abstract.strip()
            elif "\t" in line:
                parts = line.strip().split("\t")
                # Skip CID / relation lines (parts[1] == 'CID' or numeric check fails)
                if len(parts) < 6:
                    continue
                try:
                    start, end = int(parts[1]), int(parts[2])
                except ValueError:
                    continue
                etype = parts[4]
                if keep_types is not None and etype not in keep_types:
                    continue
                doc["annotations"].append(
                    {"start": start, "end": end, "text": parts[3], "type": etype}
                )

        if doc["pmid"]:
            # Offsets are over title + '\n' + abstract
            doc["text"] = doc["title"] + "\n" + doc["abstract"]
            documents.append(doc)

    return documents


# ── Load NCBI Disease (all disease subtypes) ───────────────────────────────────
NCBI_TYPES = {"SpecificDisease", "DiseaseClass", "Modifier", "CompositeMention"}

ncbi_train = parse_pubtator(NCBI_DIR / "NCBItrainset_corpus.txt",  keep_types=NCBI_TYPES)
ncbi_dev   = parse_pubtator(NCBI_DIR / "NCBIdevelopset_corpus.txt", keep_types=NCBI_TYPES)
ncbi_test  = parse_pubtator(NCBI_DIR / "NCBItestset_corpus.txt",   keep_types=NCBI_TYPES)

# ── Load BC5CDR (Disease entities only) ───────────────────────────────────────
bc5cdr_train = parse_pubtator(BC5CDR_DIR / "CDR_TrainingSet.PubTator.txt",    keep_types={"Disease"})
bc5cdr_dev   = parse_pubtator(BC5CDR_DIR / "CDR_DevelopmentSet.PubTator.txt", keep_types={"Disease"})
bc5cdr_test  = parse_pubtator(BC5CDR_DIR / "CDR_TestSet.PubTator.txt",        keep_types={"Disease"})

print(f"NCBI Disease  — train: {len(ncbi_train):4d}  dev: {len(ncbi_dev):3d}  test: {len(ncbi_test):3d}")
print(f"BC5CDR        — train: {len(bc5cdr_train):4d}  dev: {len(bc5cdr_dev):3d}  test: {len(bc5cdr_test):3d}")

NCBI Disease  — train:  593  dev: 100  test: 100
BC5CDR        — train:  500  dev: 500  test: 500


In [21]:
def dataset_stats(docs: List[Dict], name: str) -> None:
    """Print basic statistics for a list of parsed PubTator documents."""
    n_ann = sum(len(d["annotations"]) for d in docs)
    type_counts: Dict[str, int] = defaultdict(int)
    for d in docs:
        for a in d["annotations"]:
            type_counts[a["type"]] += 1

    print(f"\n{name}  ({len(docs)} docs, {n_ann} annotations)")
    for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"    {t:<25} {c:>5}")


dataset_stats(ncbi_train, "NCBI Train")
dataset_stats(ncbi_dev,   "NCBI Dev")
dataset_stats(ncbi_test,  "NCBI Test")
dataset_stats(bc5cdr_train, "BC5CDR Train")
dataset_stats(bc5cdr_dev,   "BC5CDR Dev")
dataset_stats(bc5cdr_test,  "BC5CDR Test")


NCBI Train  (593 docs, 5145 annotations)
    SpecificDisease            2972
    Modifier                   1289
    DiseaseClass                769
    CompositeMention            115

NCBI Dev  (100 docs, 787 annotations)
    SpecificDisease             412
    Modifier                    214
    DiseaseClass                126
    CompositeMention             35

NCBI Test  (100 docs, 960 annotations)
    SpecificDisease             555
    Modifier                    264
    DiseaseClass                121
    CompositeMention             20

BC5CDR Train  (500 docs, 4182 annotations)
    Disease                    4182

BC5CDR Dev  (500 docs, 4244 annotations)
    Disease                    4244

BC5CDR Test  (500 docs, 4424 annotations)
    Disease                    4424


## 2. Preprocessing: Word Tokenisation + BIO Tagging

Steps per document:
1. Reconstruct full text as `title + '\n' + abstract`.
2. Tokenise by whitespace with offset tracking (`re.finditer(r'\S+')`).
3. For each gold annotation `[start, end)`, mark the **first** overlapping word token as `B-Disease` and subsequent overlapping tokens as `I-Disease`.  
   All other tokens remain `O`.

All NCBI Disease subtypes (SpecificDisease, DiseaseClass, Modifier, CompositeMention) collapse to a single `Disease` class for training.

In [22]:
def tokenize_and_bio_tag(
    doc: Dict,
) -> Tuple[List[str], List[str], List[Tuple[int, int]]]:
    """Return (words, bio_tags, word_char_offsets) for a document.

    word_char_offsets[i] = (char_start, char_end) in doc['text'].
    bio_tags uses 'B-Disease' / 'I-Disease' / 'O'.
    """
    text = doc["text"]
    annotations = sorted(doc["annotations"], key=lambda a: a["start"])

    words, offsets = [], []
    for m in re.finditer(r"\S+", text):
        words.append(m.group())
        offsets.append((m.start(), m.end()))

    tags = ["O"] * len(words)

    for ann in annotations:
        a_start, a_end = ann["start"], ann["end"]
        first_token = True
        for i, (ws, we) in enumerate(offsets):
            if ws < a_end and we > a_start:   # overlap
                tags[i] = "B-Disease" if first_token else "I-Disease"
                first_token = False

    return words, tags, offsets


def prepare_word_level(docs: List[Dict]) -> List[Dict]:
    """Convert a list of documents to word-level dicts for HuggingFace Dataset."""
    records = []
    for doc in docs:
        words, tags, _ = tokenize_and_bio_tag(doc)
        if words:
            records.append({"words": words, "ner_tags": tags})
    return records


# Quick sanity check on the first NCBI training document
ex = ncbi_train[0]
ws, ts, offs = tokenize_and_bio_tag(ex)
print(f"Document PMID {ex['pmid']} — {len(ws)} word tokens")
print("\nAnnotations:", [(a["start"], a["end"], a["text"], a["type"]) for a in ex["annotations"][:4]])
print("\nTagged tokens (non-O):")
for w, t, (s, e) in zip(ws, ts, offs):
    if t != "O":
        print(f"  [{s:4d}:{e:4d}]  {t:<12}  '{w}'")

Document PMID 10192393 — 241 word tokens

Annotations: [(15, 26, 'skin tumour', 'DiseaseClass'), (443, 449, 'cancer', 'DiseaseClass'), (483, 496, 'colon cancers', 'DiseaseClass'), (539, 565, 'adenomatous polyposis coli', 'SpecificDisease')]

Tagged tokens (non-O):
  [  15:  19]  B-Disease     'skin'
  [  20:  26]  I-Disease     'tumour'
  [ 443: 450]  B-Disease     'cancer,'
  [ 483: 488]  B-Disease     'colon'
  [ 489: 496]  I-Disease     'cancers'
  [ 539: 550]  B-Disease     'adenomatous'
  [ 551: 560]  I-Disease     'polyposis'
  [ 561: 565]  I-Disease     'coli'
  [ 566: 572]  B-Disease     '(APC),'
  [ 670: 675]  B-Disease     'colon'
  [ 676: 679]  I-Disease     'and'
  [ 680: 684]  I-Disease     'some'
  [ 685: 690]  I-Disease     'other'
  [ 691: 698]  I-Disease     'cancers'
  [ 855: 859]  B-Disease     'skin'
  [ 860: 867]  I-Disease     'tumours'
  [ 879: 894]  B-Disease     'pilomatricomas.'
  [1021:1035]  B-Disease     'pilomatricomas'
  [1210:1214]  B-Disease     'skin'


## 3. Transformer Tokenisation + Subword BIO Alignment

Both **BERT-base** (baseline) and **BioBERT** use WordPiece tokenisation.  
`build_tokenized_datasets(model_name)` loads the correct tokenizer for each model and applies the same **first-subword rule**:

- First subword of each word → inherits the word's BIO label  
- Continuation subwords → `-100` (masked from loss, not counted in evaluation)

In [23]:
def build_tokenized_datasets(model_name: str):
    """Load the tokenizer for model_name and tokenize all NCBI + BC5CDR splits.

    Returns:
        tokenizer, tok_ncbi_train, tok_ncbi_dev, tok_ncbi_test, tok_bc5cdr_test
    """
    from transformers import BertTokenizer
    # Use BertTokenizer directly: AutoTokenizer in transformers 5.x routes all
    # models without an explicit TOKENIZER_MAPPING entry (e.g. BioBERT) through
    # TokenizersBackend, which fails for old-format checkpoints.  Both
    # bert-base-cased and biobert-base-cased-v1.1 are WordPiece models that load
    # fine with BertTokenizer.
    tok = BertTokenizer.from_pretrained(model_name)

    # Special token strings used as sentinels in the alignment below.
    _special = {tok.cls_token, tok.sep_token, tok.pad_token} - {None}

    def _align(batch: Dict) -> Dict:
        tokenized = tok(
            batch["words"],
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        all_label_ids = []
        for i, tag_seq in enumerate(batch["ner_tags"]):
            # Decode token IDs back to string tokens so we can detect
            # WordPiece continuation pieces (##) without relying on word_ids(),
            # which is not available for slow tokenizers in transformers 5.x.
            tokens = tok.convert_ids_to_tokens(tokenized["input_ids"][i])
            label_ids = []
            word_idx = -1
            for token in tokens:
                if token in _special:
                    label_ids.append(-100)          # [CLS] / [SEP] / [PAD]
                elif token.startswith("##"):
                    label_ids.append(-100)          # continuation subword
                else:
                    word_idx += 1                   # first subword of next word
                    label_ids.append(
                        LABEL2ID[tag_seq[word_idx]] if word_idx < len(tag_seq) else -100
                    )
            all_label_ids.append(label_ids)
        tokenized["labels"] = all_label_ids
        return tokenized

    def _build(records):
        return Dataset.from_list(records).map(
            _align, batched=True, remove_columns=["words", "ner_tags"]
        )

    print(f"  Tokenising with {model_name} ...")
    t_train       = _build(prepare_word_level(ncbi_train))
    t_dev         = _build(prepare_word_level(ncbi_dev))
    t_test        = _build(prepare_word_level(ncbi_test))
    t_bc5cdr_test = _build(prepare_word_level(bc5cdr_test))
    print(f"  Done — train:{len(t_train)}  dev:{len(t_dev)}  ncbi-test:{len(t_test)}  bc5cdr-test:{len(t_bc5cdr_test)}")
    return tok, t_train, t_dev, t_test, t_bc5cdr_test


print("build_tokenized_datasets() defined.")


build_tokenized_datasets() defined.


## 4. Fine-Tuning — Baseline (BERT-base) vs Model (BioBERT)

Both models are trained on **NCBI Disease** with identical hyperparameters:

| Setting | Value |
|---|---|
| Learning rate | 2e-5 |
| Batch size | 16 |
| Max epochs | 5 |
| Early stopping patience | 2 (metric = dev F1) |
| Checkpoint | saved to `./ner_model_bert-base/` and `./ner_model_biobert/` |

In [27]:
def compute_metrics(eval_pred):
    """seqeval entity-level P/R/F1 ignoring -100 masked positions."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seqs.append([ID2LABEL[l] for l in label_row if l != -100])
        pred_seqs.append([ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100])

    return {
        "precision": precision_score(true_seqs, pred_seqs),
        "recall":    recall_score(true_seqs, pred_seqs),
        "f1":        f1_score(true_seqs, pred_seqs),
    }


def build_trainer(
    model_name: str,
    tokenizer,
    tok_train,
    tok_dev,
    output_dir: str,
) -> Trainer:
    """Create a fresh model + Trainer for the given model_name."""
    from transformers import BertForTokenClassification
    # Use BertForTokenClassification directly: AutoModelForTokenClassification
    # requires a `model_type` key in config.json, which older BioBERT checkpoints
    # omit.  Both bert-base-cased and biobert-base-cased-v1.1 are BERT models.
    model = BertForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"{output_dir}/logs",
        logging_steps=50,
        report_to="none",
        save_total_limit=2,
    )
    return Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_dev,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )


print("compute_metrics() and build_trainer() defined.")


compute_metrics() and build_trainer() defined.


In [26]:
# Preserve any models already trained in this session (e.g. after a partial run).
if not isinstance(globals().get("RESULTS"), dict):
    RESULTS = {}

for model_key, model_name in MODELS.items():
    if model_key in RESULTS:
        print(f"  {model_key} already in RESULTS — skipping re-training.")
        continue

    print(f"\n{'='*60}")
    print(f"  Training: {model_key}  ({model_name})")
    print(f"{'='*60}")

    output_dir = f"./ner_model_{model_key}"

    tokenizer, tok_train, tok_dev, tok_test, tok_bc5cdr_test = \
        build_tokenized_datasets(model_name)

    trainer = build_trainer(model_name, tokenizer, tok_train, tok_dev, output_dir)

    train_result = trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    RESULTS[model_key] = {
        "model_name":       model_name,
        "output_dir":       output_dir,
        "trainer":          trainer,
        "tokenizer":        tokenizer,
        "tok_ncbi_test":    tok_test,
        "tok_bc5cdr_test":  tok_bc5cdr_test,
        "train_result":     train_result,
        "log_history":      trainer.state.log_history,
    }
    print(f"\n  {model_key} complete — train loss: {train_result.training_loss:.4f}")

print("\nAll models trained.")


  bert-base already in RESULTS — skipping re-training.

  Training: biobert  (dmis-lab/biobert-base-cased-v1.1)
  Tokenising with dmis-lab/biobert-base-cased-v1.1 ...


Map:   0%|          | 0/593 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

  Done — train:593  dev:100  ncbi-test:100  bc5cdr-test:500


ValueError: Unrecognized model in dmis-lab/biobert-base-cased-v1.1. Should have a `model_type` key in its config.json.

## 5. RQ2: Learning Curves

Plot **training loss**, **validation loss**, and **validation F1** per epoch to visualise convergence and whether early stopping triggered.

In [ ]:
MODEL_COLORS = {"bert-base": "steelblue", "biobert": "darkorange"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_key, res in RESULTS.items():
    color = MODEL_COLORS[model_key]
    log   = res["log_history"]

    eval_epochs = [e["epoch"]    for e in log if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log if "eval_loss" in e]
    eval_f1s    = [e.get("eval_f1", 0.0) for e in log if "eval_loss" in e]

    axes[0].plot(eval_epochs, eval_losses, "o-", color=color, label=model_key)
    axes[1].plot(eval_epochs, eval_f1s,    "o-", color=color, label=model_key)

for ax, title, ylabel in zip(
    axes,
    ["RQ2 — Validation Loss per Epoch", "RQ2 — Validation F1 per Epoch"],
    ["Loss", "F1 Score"],
):
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150)
plt.show()
print("Saved → learning_curves.png")

## 6. Inference on Both Datasets

Each trained model runs inference on:
- **NCBI Disease test** (in-domain)
- **BC5CDR test** (cross-domain — the transfer learning evaluation)

Subword predictions are decoded back to word-level BIO sequences by removing `-100` masked positions.

In [ ]:
def decode_predictions(trainer: Trainer, tok_dataset) -> Tuple[List, List]:
    """Run inference and return (true_seqs, pred_seqs) at word level."""
    output   = trainer.predict(tok_dataset)
    pred_ids = np.argmax(output.predictions, axis=2)
    gold_ids = tok_dataset["labels"]

    true_seqs, pred_seqs = [], []
    for pred_row, gold_row in zip(pred_ids, gold_ids):
        true_seqs.append([ID2LABEL[g] for g in gold_row if g != -100])
        pred_seqs.append([ID2LABEL[p] for p, g in zip(pred_row, gold_row) if g != -100])

    assert all(len(t) == len(p) for t, p in zip(true_seqs, pred_seqs)), \
        "Mismatch in true/pred sequence lengths!"
    return true_seqs, pred_seqs


for model_key, res in RESULTS.items():
    print(f"Inference: {model_key} ...")
    ncbi_true, ncbi_pred     = decode_predictions(res["trainer"], res["tok_ncbi_test"])
    bc5cdr_true, bc5cdr_pred = decode_predictions(res["trainer"], res["tok_bc5cdr_test"])

    RESULTS[model_key]["ncbi_true_seqs"]   = ncbi_true
    RESULTS[model_key]["ncbi_pred_seqs"]   = ncbi_pred
    RESULTS[model_key]["bc5cdr_true_seqs"] = bc5cdr_true
    RESULTS[model_key]["bc5cdr_pred_seqs"] = bc5cdr_pred
    print(f"  {model_key} done")

print("\nInference complete.")

## 7. Span Reconstruction + Entity-Level Evaluation

BIO sequences → entity spans `(start_word_idx, end_word_idx, type)`.  
Evaluation uses **exact match**: a prediction is correct only if span boundaries and entity type both match the gold annotation exactly.

In [ ]:
def bio_to_spans(tag_seq: List[str]) -> Set[Tuple[int, int, str]]:
    """Convert a BIO label sequence to a set of (start, end, type) spans.

    'end' is exclusive. Broken I-* tags (no preceding B) are treated as B.
    """
    spans: Set[Tuple[int, int, str]] = set()
    start, current_type = None, None

    for i, tag in enumerate(tag_seq):
        if tag.startswith("B-"):
            if start is not None:
                spans.add((start, i, current_type))
            start, current_type = i, tag[2:]
        elif tag.startswith("I-"):
            if start is None or tag[2:] != current_type:
                if start is not None:
                    spans.add((start, i, current_type))
                start, current_type = i, tag[2:]
        else:
            if start is not None:
                spans.add((start, i, current_type))
            start, current_type = None, None

    if start is not None:
        spans.add((start, len(tag_seq), current_type))
    return spans


def entity_level_prf(true_seqs: List[List[str]], pred_seqs: List[List[str]]) -> Dict:
    """Exact entity-level Precision / Recall / F1."""
    tp = fp = fn = 0
    for ts, ps in zip(true_seqs, pred_seqs):
        gold  = bio_to_spans(ts)
        preds = bio_to_spans(ps)
        tp += len(gold & preds)
        fp += len(preds - gold)
        fn += len(gold - preds)
    P  = tp / (tp + fp) if tp + fp else 0.0
    R  = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    return {"precision": P, "recall": R, "f1": F1, "tp": tp, "fp": fp, "fn": fn}


# ── Compute metrics for all model × dataset combinations ──────────────────────
all_metrics: Dict[Tuple[str, str], Dict] = {}

print(f"{'Model':<12}  {'Dataset':<22}  {'P':>7}  {'R':>7}  {'F1':>7}  {'TP':>6}  {'FP':>6}  {'FN':>6}")
print("-" * 80)
for model_key, res in RESULTS.items():
    for dataset_name, true_seqs, pred_seqs in [
        ("NCBI Disease (in-domain)",  res["ncbi_true_seqs"],   res["ncbi_pred_seqs"]),
        ("BC5CDR (cross-domain)",     res["bc5cdr_true_seqs"],  res["bc5cdr_pred_seqs"]),
    ]:
        m = entity_level_prf(true_seqs, pred_seqs)
        all_metrics[(model_key, dataset_name)] = m
        print(f"{model_key:<12}  {dataset_name:<22}  "
              f"{m['precision']:>7.4f}  {m['recall']:>7.4f}  {m['f1']:>7.4f}  "
              f"{m['tp']:>6}  {m['fp']:>6}  {m['fn']:>6}")

# ── Bar chart: F1 comparison ───────────────────────────────────────────────────
datasets_short = ["NCBI Disease (in-domain)", "BC5CDR (cross-domain)"]
x     = np.arange(len(datasets_short))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (model_key, color) in enumerate(MODEL_COLORS.items()):
    f1s  = [all_metrics[(model_key, d)]["f1"] for d in datasets_short]
    bars = axes[0].bar(x + (i - 0.5) * width, f1s, width,
                       label=model_key, color=color, alpha=0.85)
    for bar, v in zip(bars, f1s):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.005, f"{v:.3f}",
                     ha="center", va="bottom", fontsize=9)

axes[0].set_xticks(x)
axes[0].set_xticklabels(["NCBI Disease\n(in-domain)", "BC5CDR\n(cross-domain)"])
axes[0].set_ylabel("Entity-Level F1")
axes[0].set_ylim(0, 1)
axes[0].set_title("F1: BERT-base vs BioBERT")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# P / R / F1 on BC5CDR (cross-domain transfer)
metrics_names  = ["precision", "recall", "f1"]
x2 = np.arange(len(metrics_names))
for i, (model_key, color) in enumerate(MODEL_COLORS.items()):
    vals = [all_metrics[(model_key, "BC5CDR (cross-domain)")][m] for m in metrics_names]
    axes[1].bar(x2 + (i - 0.5) * width, vals, width,
                label=model_key, color=color, alpha=0.85)

axes[1].set_xticks(x2)
axes[1].set_xticklabels(["Precision", "Recall", "F1"])
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].set_title("BC5CDR Transfer — P/R/F1: BERT-base vs BioBERT")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("comparison.png", dpi=150)
plt.show()
print("Saved → comparison.png")

# seqeval reports
for model_key, res in RESULTS.items():
    print(f"\n── seqeval: {model_key} on BC5CDR ──────────────────────")
    print(classification_report(res["bc5cdr_true_seqs"], res["bc5cdr_pred_seqs"]))

## 8. RQ1: Per-Category Analysis

For the **NCBI Disease test set** we know the original annotation subtype of every gold span (SpecificDisease, DiseaseClass, Modifier, CompositeMention).  
We measure **Recall per subtype** for both models — which disease categories does BioBERT handle better than BERT-base?

In [ ]:
def per_subtype_recall(docs: List[Dict], pred_seqs: List[List[str]]) -> Dict[str, Dict]:
    """Recall per original NCBI annotation subtype (exact word-span match)."""
    assert len(docs) == len(pred_seqs)
    counts: Dict[str, Dict[str, int]] = defaultdict(lambda: {"tp": 0, "fn": 0})

    for doc, pred_seq in zip(docs, pred_seqs):
        words, _, word_offsets = tokenize_and_bio_tag(doc)
        pred_word_spans: Set[Tuple[int, int]] = {
            (s, e) for s, e, _ in bio_to_spans(pred_seq)
        }
        for ann in doc["annotations"]:
            a_s, a_e = ann["start"], ann["end"]
            word_idxs = [i for i, (ws, we) in enumerate(word_offsets)
                         if ws < a_e and we > a_s]
            if not word_idxs:
                continue
            gold_span = (word_idxs[0], word_idxs[-1] + 1)
            subtype   = ann["type"]
            if gold_span in pred_word_spans:
                counts[subtype]["tp"] += 1
            else:
                counts[subtype]["fn"] += 1

    results = {}
    for stype, c in counts.items():
        tp, fn = c["tp"], c["fn"]
        total  = tp + fn
        results[stype] = {"tp": tp, "fn": fn, "total": total,
                          "recall": tp / total if total else 0.0}
    return results


# ── Compute for both models ────────────────────────────────────────────────────
rq1_results: Dict[str, Dict] = {}
for model_key, res in RESULTS.items():
    assert len(ncbi_test) == len(res["ncbi_pred_seqs"]), \
        f"Length mismatch for {model_key}"
    rq1_results[model_key] = per_subtype_recall(ncbi_test, res["ncbi_pred_seqs"])

# ── Table ──────────────────────────────────────────────────────────────────────
all_subtypes = sorted(
    {s for r in rq1_results.values() for s in r},
    key=lambda s: -list(rq1_results.values())[0].get(s, {}).get("total", 0)
)
header = f"{'Subtype':<25}" + "".join(f"  {mk:<12}" for mk in MODELS)
print("RQ1 — Recall by annotation subtype (NCBI Disease test set)")
print(header)
print("-" * len(header))
for stype in all_subtypes:
    row = f"{stype:<25}"
    for model_key in MODELS:
        r = rq1_results[model_key].get(stype, {})
        rc = r.get("recall", 0.0)
        n  = r.get("total", 0)
        row += f"  {rc:.4f} (n={n:<4})"
    print(row)

# ── Grouped bar chart ──────────────────────────────────────────────────────────
x     = np.arange(len(all_subtypes))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))

for i, (model_key, color) in enumerate(MODEL_COLORS.items()):
    recalls = [rq1_results[model_key].get(s, {}).get("recall", 0.0) for s in all_subtypes]
    totals  = [rq1_results[model_key].get(s, {}).get("total",  0)   for s in all_subtypes]
    bars = ax.bar(x + (i - 0.5) * width, recalls, width,
                  label=model_key, color=color, alpha=0.85)
    if i == 0:  # annotate n= once
        for bar, n in zip(bars, totals):
            ax.text(bar.get_x() + width, bar.get_height() + 0.01,
                    f"n={n}", ha="center", va="bottom", fontsize=8, color="gray")

ax.set_xticks(x)
ax.set_xticklabels(all_subtypes, rotation=15, ha="right")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1.15)
ax.set_title("RQ1 — Recall per disease subtype: BERT-base vs BioBERT")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("rq1_per_category.png", dpi=150)
plt.show()

## 9. RQ3: Error Analysis

Errors are categorised into three types:

| Category | Definition |
|---|---|
| **False Positive** | Predicted span has no overlap with any gold span |
| **False Negative** | Gold span was entirely missed by predictions |
| **Partial Match** | Predicted and gold span overlap but boundaries differ |

Sample examples with local word context are shown for each category.

In [ ]:
def error_analysis(
    docs: List[Dict],
    pred_seqs: List[List[str]],
    n_examples: int = 5,
    dataset_name: str = "",
) -> Dict[str, int]:
    """Categorise errors as FP / FN / Partial Match. Returns counts."""
    false_positives, false_negatives, partial_matches = [], [], []

    for doc, pred_seq in zip(docs, pred_seqs):
        words, true_tags, word_offsets = tokenize_and_bio_tag(doc)

        gold_spans  = bio_to_spans(true_tags)
        pred_spans  = bio_to_spans(pred_seq)
        gold_ranges = {(s, e) for s, e, _ in gold_spans}
        pred_ranges = {(s, e) for s, e, _ in pred_spans}
        ctx = lambda s, e: " ".join(words[max(0, s - 3): e + 3])

        for ps, pe, _ in pred_spans:
            if (ps, pe) not in gold_ranges:
                overlaps = any(ps < ge and pe > gs for gs, ge in gold_ranges)
                entry = {"pmid": doc["pmid"], "text": " ".join(words[ps:pe]),
                         "context": ctx(ps, pe)}
                (partial_matches if overlaps else false_positives).append(entry)

        for gs, ge, _ in gold_spans:
            if (gs, ge) not in pred_ranges:
                false_negatives.append({
                    "pmid": doc["pmid"], "text": " ".join(words[gs:ge]),
                    "context": ctx(gs, ge),
                })

    label = f" [{dataset_name}]" if dataset_name else ""
    print(f"\nRQ3 — Error Analysis{label}")
    print(f"  False Positives (hallucinated)  : {len(false_positives):5d}")
    print(f"  False Negatives (missed)        : {len(false_negatives):5d}")
    print(f"  Partial Matches (wrong boundary): {len(partial_matches):5d}")

    def _show(items, title):
        print(f"\n  ── {title} (top {n_examples}) ──")
        for ex in items[:n_examples]:
            print(f"    PMID {ex['pmid']}: '{ex['text']}'")
            print(f"      context: ...{ex['context']}...")

    _show(false_positives, "False Positives")
    _show(false_negatives, "False Negatives")
    _show(partial_matches, "Partial Matches")

    return {"FP": len(false_positives), "FN": len(false_negatives),
            "Partial": len(partial_matches)}


# ── Run error analysis for both models on BC5CDR (cross-domain) ───────────────
rq3_counts: Dict[str, Dict] = {}
for model_key, res in RESULTS.items():
    assert len(bc5cdr_test) == len(res["bc5cdr_pred_seqs"])
    rq3_counts[model_key] = error_analysis(
        bc5cdr_test, res["bc5cdr_pred_seqs"],
        n_examples=3, dataset_name=f"{model_key} / BC5CDR"
    )

# ── Grouped bar chart ──────────────────────────────────────────────────────────
categories = ["FP", "FN", "Partial"]
x     = np.arange(len(categories))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))

for i, (model_key, color) in enumerate(MODEL_COLORS.items()):
    vals = [rq3_counts[model_key][c] for c in categories]
    bars = ax.bar(x + (i - 0.5) * width, vals, width,
                  label=model_key, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1, str(v),
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(["False Positives\n(hallucinated)",
                    "False Negatives\n(missed)",
                    "Partial Matches\n(boundary error)"])
ax.set_ylabel("Count")
ax.set_title("RQ3 — Error distribution on BC5CDR: BERT-base vs BioBERT")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("rq3_errors_bc5cdr.png", dpi=150)
plt.show()